# Cervical cytology classification with machine learning

**Research question.** Can edge and color features distinguish five cell types in cropped cervical Pap smear images when related crops are kept together during evaluation?

[SIPaKMeD on Kaggle](https://www.kaggle.com/datasets/prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed) · [Original dataset](https://www.cs.uoi.gr/~marina/sipakmed.html) · [Plissiti et al., ICIP 2018](https://www.cse.uoi.gr/~sfikas/sipakmed2018.pdf)

This is a cell-type benchmark for cervical cancer screening research. The five labels do not include confirmed invasive cancer.

## 1. Download and index the Kaggle dataset

Run on a Colab CPU runtime. The class and parent image identifiers are used to group related cell crops. The parent identifier is derived from the crop filename, and each proposed group must match a source image within its class archive directory.

In [ ]:
%pip -q install kagglehub

In [ ]:
import hashlib
import platform
import re
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from PIL import Image
from skimage.feature import hog
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
                             classification_report, f1_score)
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
HANDLE = "prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed"
root = Path(kagglehub.dataset_download(HANDLE))
print("Dataset cache:", root)
print("Python:", platform.python_version(), "| scikit-learn:", sklearn.__version__)
expected = {"Dyskeratotic": 813, "Koilocytotic": 825, "Metaplastic": 793,
            "Parabasal": 787, "Superficial-Intermediate": 831}
labels = list(expected)
rows = []
for path in sorted(root.rglob("*")):
    if path.suffix.lower() != ".bmp" or path.parent.name.upper() != "CROPPED":
        continue
    label = path.parent.parent.name.removeprefix("im_")
    match = re.fullmatch(r"(\d+)_(\d+)", path.stem)
    if label not in expected or match is None:
        raise ValueError(f"Unexpected crop layout: {path.relative_to(root)}")
    parent_id = match.group(1)
    parent = path.parent.parent / f"{parent_id}.bmp"
    if not parent.is_file():
        raise ValueError(f"Missing cluster image for {path.relative_to(root)}")
    rows.append({"path": path, "label": label, "group": f"{label}/{parent_id}"})
if not rows:
    raise ValueError("No cropped BMPs found in this Kaggle dataset version.")
samples = pd.DataFrame(rows)
counts = samples["label"].value_counts().to_dict()
assert counts == expected, counts
print("Parent clusters:", samples["group"].nunique())
samples["sha256"] = [hashlib.sha256(p.read_bytes()).hexdigest() for p in samples["path"]]
display(samples.groupby("label").agg(crops=("path", "size"), clusters=("group", "nunique")))
print("Repeated file hashes:", samples["sha256"].duplicated().sum())

## 2. Split by parent cluster and extract image features

The first stratified group fold serves as the test set. Every crop from a parent cluster stays on the same side of a split. The notebook also checks for exact file duplicates across partitions. Fixed per-image transformations learn nothing from other images.

In [ ]:
y = samples["label"].to_numpy()
groups = samples["group"].to_numpy()
hashes = samples["sha256"].to_numpy()
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train, test = next(splitter.split(samples, y, groups))

def check_split(left, right):
    assert set(groups[left]).isdisjoint(groups[right]), "Parent cluster leakage"
    assert set(hashes[left]).isdisjoint(hashes[right]), "Exact file duplicate across partitions"
    assert set(y[left]) == set(y[right]) == set(labels), "Missing class in a partition"

check_split(train, test)
print("Training crops:", len(train), "| Holdout crops:", len(test))
fig, axes = plt.subplots(1, len(labels), figsize=(15, 3))
training = samples.iloc[train]
for label, ax in zip(labels, axes):
    path = training.loc[training["label"].eq(label), "path"].iloc[0]
    with Image.open(path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(label.replace("-", "-\n"))
    ax.axis("off")
fig.tight_layout()
plt.show()

def image_features(path):
    with Image.open(path) as source:
        rgb = source.convert("RGB").resize((64, 64), Image.Resampling.BILINEAR)
    gray = np.asarray(rgb.convert("L"))
    edges = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                cells_per_block=(2, 2), block_norm="L2-Hys")
    pixels = np.asarray(rgb)
    color_counts = [np.histogram(pixels[:, :, i], bins=16, range=(0, 256))[0] / gray.size
                    for i in range(3)]
    return np.concatenate([edges, *color_counts]).astype(np.float32)

X = np.stack([image_features(path) for path in samples["path"]])
assert X.shape == (len(samples), 1812) and np.isfinite(X).all()
print("Feature matrix:", X.shape)

## 3. Compare models using training folds

Macro F1 averages the per-class F1 scores equally. The dummy classifier provides a majority-class baseline. Logistic regression uses fold-local scaling. The random forest can learn nonlinear combinations of edge and color features.

In [ ]:
models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Random forest": RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=2),
}
inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
folds = list(inner.split(X[train], y[train], groups[train]))
for fit, valid in folds:
    check_split(train[fit], train[valid])
rows = []
for name, model in models.items():
    scores = cross_validate(model, X[train], y[train], cv=folds,
                            scoring=["f1_macro", "balanced_accuracy"], error_score="raise")
    rows.append({"Model": name, "Mean macro F1": scores["test_f1_macro"].mean(),
                 "Macro F1 fold SD": scores["test_f1_macro"].std(ddof=1),
                 "Mean balanced accuracy": scores["test_balanced_accuracy"].mean()})
comparison = pd.DataFrame(rows).sort_values("Mean macro F1", ascending=False, kind="stable")
display(comparison.round(4))
winner = comparison.iloc[0]["Model"]
print("Selected:", winner)

## 4. Evaluate on the held-out clusters

Inspect per-class recall and the most frequent confusion before making claims about performance. Cell crops within a cluster can be correlated, so the crop count alone does not establish statistical precision.

In [ ]:
model = models[winner].fit(X[train], y[train])
prediction = model.predict(X[test])
summary = pd.Series({"Accuracy": accuracy_score(y[test], prediction),
                     "Balanced accuracy": balanced_accuracy_score(y[test], prediction),
                     "Macro F1": f1_score(y[test], prediction, average="macro")})
display(summary.round(4).rename("Holdout"))
report = classification_report(y[test], prediction, labels=labels, output_dict=True, zero_division=0)
display(pd.DataFrame(report).T.round(4))
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(y[test], prediction, labels=labels, ax=ax,
                                        xticks_rotation=35, colorbar=False)
fig.tight_layout()
plt.show()
errors = samples.iloc[test][["path", "label", "group"]].copy()
errors["prediction"] = prediction
errors["path"] = errors["path"].map(lambda path: str(path.relative_to(root)))
display(errors.loc[errors["label"].ne(errors["prediction"])].head(15))
matrix = (
    pd.crosstab(y[test], prediction)
    .reindex(index=labels, columns=labels, fill_value=0)
    .rename_axis(index="Actual", columns="Predicted")
)
print(matrix.to_string())

## 5. Interpret the benchmark

Report the winning training cross-validation score alongside the held-out macro F1, class support, and most common confusion. Check whether the models improve on the baseline. Parent grouping addresses related crops from a cluster. Patient-level separation would require patient identifiers. The five labels represent cell morphology, and results do not establish screening performance or accuracy for invasive cancer.

Possible follow-ups include replacing HOG and color histograms with pretrained image embeddings, and evaluating generalization with a separately collected, appropriately labeled cohort.